In [5]:
import pandas as pd
from scipy.stats import shapiro, pearsonr, spearmanr

# 1. Chargement des données
df_h3 = pd.read_csv("https://raw.githubusercontent.com/lunettenoire/wellbeing_project/refs/heads/main/data/cleaned/df_h3.csv")

alpha = 0.05

# 2. Test de normalité initial pour 'life_ladder' (échantillonné à 5000 lignes max si nécessaire)
df_shapiro = df_h3.sample(min(5000, len(df_h3)), random_state=42) if len(df_h3) > 5000 else df_h3
_, p_life = shapiro(df_shapiro['life_ladder'].dropna())


# 3. Fonction pour automatiser le calcul, l'interprétation et le formatage du print
def analyser_et_afficher_correlation(df, col_y, col_x, p_life_normalite):
    # On retire les lignes contenant des NaNs uniquement pour ce couple de variables
    df_clean = df[[col_y, col_x]].dropna()
    
    # Test de normalité pour la variable X
    df_shapiro_x = df_clean.sample(min(5000, len(df_clean)), random_state=42) if len(df_clean) > 5000 else df_clean
    _, p_x = shapiro(df_shapiro_x[col_x])
    
    # Choix de la méthode selon le test de normalité
    if p_life_normalite > alpha and p_x > alpha:
        method_name = 'pearson'
        symbol = 'r'
        coef, p_val_test = pearsonr(df_clean[col_y], df_clean[col_x])
    else:
        method_name = 'spearman'
        symbol = 'rho'
        coef, p_val_test = spearmanr(df_clean[col_y], df_clean[col_x])
        
    # Logique d'interprétation automatique (Seuils standards : 0.7 très fort, 0.3 modéré)
    abs_coef = abs(coef)
    if coef >= 0:
        if abs_coef >= 0.7:
            interpretation = f"LIEN POSITIF TRÈS FORT. Plus '{col_x}' augmente, plus le bonheur augmente."
        elif abs_coef >= 0.3:
            interpretation = f"LIEN POSITIF MODÉRÉ. Plus '{col_x}' augmente, plus le bonheur tend à augmenter."
        else:
            interpretation = f"LIEN POSITIF FAIBLE. Légère tendance à la hausse du bonheur."
    else:
        if abs_coef >= 0.7:
            interpretation = f"LIEN NÉGATIF TRÈS FORT. Plus '{col_x}' augmente, plus le bonheur diminue."
        elif abs_coef >= 0.3:
            interpretation = f"LIEN NÉGATIF MODÉRÉ. Plus '{col_x}' augmente, plus le bonheur diminue."
        else:
            interpretation = f"LIEN NÉGATIF FAIBLE. Légère tendance à la baisse du bonheur."
            
    # Affichage respectant exactement votre structure
    print(f"• Lien entre '{col_y}' et '{col_x}' :")
    print(f"  - Coefficient de corrélation ({symbol}) : {coef:.3f}")
    print(f"  - P-value du test : {p_val_test:.3e}")
    print(f"  -> Interprétation : {interpretation}")
    print("-" * 60)

# 4. Exécution pour vos deux variables
analyser_et_afficher_correlation(df_h3, 'life_ladder', 'positive_affect', p_life)
analyser_et_afficher_correlation(df_h3, 'life_ladder', 'negative_affect', p_life)

• Lien entre 'life_ladder' et 'positive_affect' :
  - Coefficient de corrélation (rho) : 0.488
  - P-value du test : 4.633e-67
  -> Interprétation : LIEN POSITIF MODÉRÉ. Plus 'positive_affect' augmente, plus le bonheur tend à augmenter.
------------------------------------------------------------
• Lien entre 'life_ladder' et 'negative_affect' :
  - Coefficient de corrélation (rho) : -0.494
  - P-value du test : 6.203e-69
  -> Interprétation : LIEN NÉGATIF MODÉRÉ. Plus 'negative_affect' augmente, plus le bonheur diminue.
------------------------------------------------------------
